In [0]:
# =============================================================================
# active_archive_reconcile  -  READ ONLY. Case-by-case reconciliation ACROSS active + archive,
# to prove NO DUPLICATION (a case in both) and NO MISSING (an eligible case in neither) after the
# 2311/2307 segmentation changes. Run BEFORE and AFTER the change and compare.
#   ACTIVE  = stg_segmentation_states (CaseNo -> TargetState)
#   ARCHIVE = the archive segmentation result (CaseNo -> Segment: ARIAFTA/ARIAUTA/ARIAFPA)
#   UNIVERSE (optional) = the eligible case list (to find cases in NEITHER = a gap)
# Reports: OVERLAP (in both), active-only, archive-only, NEITHER, + per-case samples. Single print.
# =============================================================================

In [0]:
# ---- CELL 0 : config + auth ----
ACTIVE_TBL  = "hive_metastore.ariadm_active_appeals.stg_segmentation_states"   # CaseNo + TargetState
ARCHIVE_SEGMENTS = [   # canonical archive appeals set = union of FTA/UTA/FPA (from Overall/1_OVERALL_RECONCILIATION_TESTS)
 ("FTA", "hive_metastore.ariadm_arm_fta.stg_appeals_filtered"),
 ("UTA", "hive_metastore.ariadm_arm_uta.stg_appeals_filtered"),
 ("FPA", "hive_metastore.ariadm_arm_fpa.stg_filepreservedcases_filtered"),
]
UNIVERSE_TBL= ""     # <-- OPTIONAL: eligible-case universe (CaseNo) to detect NEITHER/gap. Leave "" to skip.
                     #     e.g. a bronze appealcase table filtered to DeptId NOT IN (519,520).

from pyspark.sql import functions as F
from pyspark.sql.functions import *
REPORT=[]
def log(*a): REPORT.append(" ".join(str(x) for x in a))
def logdf(df,n=30):
    try: REPORT.append(df._jdf.showString(n,0,False))
    except Exception as e: REPORT.append(f"  (render fail: {str(e)[:100]})")
_c=spark.read.option("multiline","true").json("dbfs:/configs/config.json")
env_name=_c.first()["env"].strip().lower(); lz_key=_c.first()["lz_key"].strip().lower()
KV=f"ingest{lz_key}-meta002-{env_name}"
cid=dbutils.secrets.get(KV,"SERVICE-PRINCIPLE-CLIENT-ID"); csec=dbutils.secrets.get(KV,"SERVICE-PRINCIPLE-CLIENT-SECRET"); tid=dbutils.secrets.get(KV,"SERVICE-PRINCIPLE-TENANT-ID")
for sa in [f"ingest{lz_key}curated{env_name}",f"ingest{lz_key}raw{env_name}"]:
    spark.conf.set(f"fs.azure.account.auth.type.{sa}.dfs.core.windows.net","OAuth")
    spark.conf.set(f"fs.azure.account.oauth.provider.type.{sa}.dfs.core.windows.net","org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    spark.conf.set(f"fs.azure.account.oauth2.client.id.{sa}.dfs.core.windows.net",cid)
    spark.conf.set(f"fs.azure.account.oauth2.client.secret.{sa}.dfs.core.windows.net",csec)
    spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{sa}.dfs.core.windows.net",f"https://login.microsoftonline.com/{tid}/oauth2/token")

def col_ci(cols,name): return next((c for c in cols if c.lower()==name.lower()),None)

In [0]:
# ---- CELL 1 : load ACTIVE + ARCHIVE (auto-discover archive if not set) ----
log("="*74); log("ACTIVE + ARCHIVE reconciliation"); log("="*74)
A=spark.table(ACTIVE_TBL); acc=col_ci(A.columns,"CaseNo"); atc=col_ci(A.columns,"TargetState")
active=A.select(trim(col(acc)).alias("CaseNo"), (col(atc) if atc else lit(None)).alias("TargetState")).distinct()
log(f"ACTIVE  {ACTIVE_TBL}: {active.count()} distinct CaseNos")

arch_parts=[]
for seg_label, tbl in ARCHIVE_SEGMENTS:
    try:
        AR=spark.table(tbl); arc=col_ci(AR.columns,"CaseNo"); asg=col_ci(AR.columns,"Segment")
        if not arc: log(f"  ARCHIVE {seg_label}: no CaseNo in {tbl}, skip"); continue
        part=AR.select(trim(col(arc)).alias("CaseNo"), (col(asg).cast("string") if asg else lit("ARIA"+seg_label)).alias("Segment"))
        arch_parts.append(part); log(f"  {seg_label}: {part.select('CaseNo').distinct().count()} distinct from {tbl}")
    except Exception as e: log(f"  ARCHIVE {seg_label} ERR {str(e)[:100]}")
if not arch_parts:
    log("!! no archive segment tables readable - check ARCHIVE_SEGMENTS."); print("\n".join(REPORT)); dbutils.notebook.exit("no archive")
archive=arch_parts[0]
for p in arch_parts[1:]: archive=archive.unionByName(p)
archive=archive.dropDuplicates(["CaseNo"])
log(f"ARCHIVE (FTA+UTA+FPA union): {archive.count()} distinct CaseNos")

In [0]:
# ---- CELL 2 : the reconciliation - OVERLAP (dup), active-only, archive-only ----
both     = active.join(archive, "CaseNo", "inner")
act_only = active.join(archive.select("CaseNo"), "CaseNo", "left_anti")
arc_only = archive.join(active.select("CaseNo"), "CaseNo", "left_anti")
nB=both.count()
log("\n"+"-"*74)
log(f"IN BOTH (DUPLICATION / OVERLAP): {nB}   <-- should be 0")
log(f"ACTIVE only:  {act_only.count()}")
log(f"ARCHIVE only: {arc_only.count()}")
if nB:
    log("\n>>> OVERLAP cases (a case migrated to BOTH active and archive) - by state x segment:")
    logdf(both.groupBy("TargetState","Segment").count().orderBy(desc("count")), 40)
    log("\nsample overlap CaseNos:")
    logdf(both.select("CaseNo","TargetState","Segment").limit(40))
log("\nactive breakdown by TargetState:")
logdf(active.groupBy("TargetState").count().orderBy(desc("count")),40)
log("archive breakdown by Segment:")
logdf(archive.groupBy("Segment").count().orderBy(desc("count")),40)

# per-segment overlap: active vs EACH archive segment individually, incl TD (which the
# FTA/UTA/FPA union above excludes). Matches the Overall recon active_vs_<seg> checks.
TD_TBL = "hive_metastore.ariadm_arm_td.stg_td_filtered"
SEG_ALL = list(ARCHIVE_SEGMENTS) + [("TD", TD_TBL)]
log("\n--- per-segment overlap: active INTERSECT each archive segment (incl TD) ---")
for seg_label, tbl in SEG_ALL:
    try:
        AR=spark.table(tbl); arc=col_ci(AR.columns,"CaseNo")
        segc=AR.select(trim(col(arc)).alias("CaseNo")).distinct()
        ov=active.select("CaseNo","TargetState").join(segc,"CaseNo","inner")
        n=ov.count()
        log(f"  active INTERSECT {seg_label}: {n}" + ("   <-- OVERLAP" if n>0 else "   ok (disjoint)"))
        if n>0: logdf(ov.groupBy("TargetState").count().orderBy(desc("count")),20)
    except Exception as e: log(f"  {seg_label} ERR {str(e)[:80]}")

In [0]:
# ---- CELL 3 : NEITHER (missing/gap) - only if a UNIVERSE of eligible cases is provided ----
log("\n"+"-"*74)
if UNIVERSE_TBL:
    U=spark.table(UNIVERSE_TBL); ucc=col_ci(U.columns,"CaseNo")
    uni=U.select(trim(col(ucc)).alias("CaseNo")).distinct()
    placed=active.select("CaseNo").union(archive.select("CaseNo")).distinct()
    neither=uni.join(placed,"CaseNo","left_anti")
    log(f"UNIVERSE {UNIVERSE_TBL}: {uni.count()} eligible cases")
    log(f"IN NEITHER active nor archive (MISSING/GAP): {neither.count()}   <-- should be 0 (or = intended left-behind)")
    if neither.count(): logdf(neither.limit(50))
else:
    log("NEITHER/gap check SKIPPED (set UNIVERSE_TBL to the eligible-case list to detect cases in neither).")

In [0]:
# ---- CELL 4 : verdict + single print ----
log("\n"+"="*74)
verdict = "PASS - no overlap (0 in both)" if nB==0 else f"FAIL - {nB} case(s) in BOTH active and archive (duplication)"
log(f">>> OVERLAP VERDICT: {verdict}")
log("READ: run this BEFORE and AFTER the 2311/2307 change. Overlap must stay 0. active-only + archive-only")
log("      should = the eligible universe (no gap). Any case moving active<->archive appears in the diff of")
log("      the two runs. Pair with segmentation_snapshot/diff for the per-case active-state movement.")
full="\n".join(REPORT)
try:
    from datetime import datetime
    user=spark.sql("SELECT current_user()").first()[0]; ts=datetime.now().strftime("%Y%m%d_%H%M%S")
    folder=f"/Workspace/Users/{user}/Results/active_archive_reconcile/{ts}"; dbutils.fs.mkdirs(f"file:{folder}")
    p=f"{folder}/active_archive_reconcile.txt"; open(p,"w").write(full); full+=f"\n\n>>> saved to: {p}"
except Exception as e: full+=f"\n(save failed: {str(e)[:80]})"
print(full)